In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import torch
from scripts.utils.load_model import load_model, ModelName
from scripts.utils.load_dataset import load_dataset


model, dataset_transform = load_model(ModelName.RESNET50, dataset_name="imagenet", device="cuda:0")

In [ ]:
model

In [ ]:
dl_train, dl_eval, dl_test = load_dataset("imagenet", transform=dataset_transform, train_batch=128, eval_batch=128)

In [ ]:
import torchattacks

attk_eps = 8 / 255
attk_alpha = 8 / 255
attk_steps = 5

# torch_attack = torchattacks.FGSM(model, eps=attk_eps) # very fast; good
# torch_attack = torchattacks.FFGSM(model, eps=attk_eps, alpha=attk_alpha)  # very fast; very good
# torch_attack = torchattacks.DIFGSM(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # fast
# torch_attack = torchattacks.RFGSM(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # fast; very good
# torch_attack = torchattacks.BIM(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # fast
# torch_attack = torchattacks.TIFGSM(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # fast
# torch_attack = torchattacks.MIFGSM(model, eps=attk_eps, alpha=attl_alpha, steps=attk_steps)  # fast
# torch_attack = torchattacks.NIFGSM(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # fast
# torch_attack = torchattacks.PGDL2(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # fast
# torch_attack = torchattacks.PGD(model) # fast
# torch_attack = torchattacks.TPGD(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # fast
# torch_attack = torchattacks.UPGD(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps, loss="dlr") # fast; very good
# torch_attack = torchattacks.APGD(model, eps=attk_eps, steps=attk_steps, loss="dlr") # fast; good
# torch_attack = torchattacks.Jitter(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # fast; good
# torch_attack = torchattacks.PIFGSM(model, max_epsilon=attk_eps, num_iter_set=attk_steps) # normal
# torch_attack = torchattacks.PIFGSMPP(model, max_epsilon=attk_eps, num_iter_set=attk_steps) # normal
# torch_attack = torchattacks.PGDRS(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # normal
# torch_attack = torchattacks.EOTPGD(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # normal
# torch_attack = torchattacks.PGDRSL2(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # slow
# torch_attack = torchattacks.SINIFGSM(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # slow
# torch_attack = torchattacks.VMIFGSM(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # very slow
# torch_attack = torchattacks.EADEN(model, lr=attk_alpha, max_iterations=attk_steps) # very slow
# torch_attack = torchattacks.VNIFGSM(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # very slow
# torch_attack = torchattacks.APGDT(model, eps=attk_eps, steps=attk_steps) # very slow
# torch_attack = torchattacks.SPSA(model, eps=attk_eps, lr=attk_alpha) # very slow
# torch_attack = torchattacks.FAB(model, eps=attk_eps, steps=attk_steps) # very slow
# torch_attack = torchattacks.CW(model, steps=attk_steps, lr=attk_alpha) # very slow
# torch_attack = torchattacks.AutoAttack(model, eps=attk_eps) # super slow
# torch_attack = torchattacks.Square(model, eps=attk_eps) # super slow
# torch_attack = torchattacks.DeepFool(model, steps=attk_steps) # super slow

In [ ]:
# torch_attack = torchattacks.TPGD(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # fast
# torch_attack = torchattacks.APGD(model, eps=attk_eps, steps=attk_steps, loss="dlr") # fast; good
# torch_attack = torchattacks.Jitter(model, eps=attk_eps, alpha=attk_alpha, steps=attk_steps) # fast; good
# torch_attack = torchattacks.FAB(model, eps=attk_eps, steps=attk_steps, multi_targeted=True)  # very slow
# torchattacks.AutoAttack

In [ ]:
import torchattacks.attack
from ulib import StopCriteria, PertModule
from ulib.attacks.iml_uap import IML_UAP, CosSim
from ulib.activation_extractor import ActivationExtractor

stop = StopCriteria(max_epochs=30)
pert_model = PertModule(model, data_shape=dl_train.get_tensor(0).shape[1:], eps=16 / 255, random_init=True)
optim = torch.optim.Adam(pert_model.parameters(), lr=0.1)
criterion = CosSim(reduce_fn=lambda losses: torch.sum(losses, dim=-1).mean())
activ_extractor = ActivationExtractor(
    model,
    "model.avgpool",
    exact_match=True,
    capture_output=True,
)


def attack_builder(model: torch.nn.Module, epoch: int) -> torchattacks.attack.Attack:
    # eps = (pert_model.eps / stop.max_epochs) * (epoch + 1)
    eps = pert_model.eps
    # eps= 0.1
    # return torchattacks.FGSM(model, eps=eps)
    # return torchattacks.FFGSM(model, eps=eps, alpha=eps)
    return torchattacks.PGD(model, eps=eps, alpha=eps / 3, steps=5)
    # return torchattacks.Jitter(model, eps=eps, alpha=eps / 3, steps=5)
    # return torchattacks.CW(model, steps=5, lr=eps)

dl_train.tensors[1].fill_(0)  # set all labels to the same class for targeted attack
dl_eval.tensors[1].fill_(0)  # set all labels to the same class for targeted attack
dl_test.tensors[1].fill_(0)  # set all labels to the same class for targeted attack

attack = IML_UAP(
    pert_model=pert_model,
    optimizer=optim,
    inner_attack=attack_builder,
    criterion=criterion,
    activ_extractor=activ_extractor,
    mixed_precision=True,
    skip_already_fooled=True,
    skip_failed_attacks=True,
    targeted=True,
)

pert = attack.fit(dl_train, dl_eval, stop)

In [ ]:
from ulib.evaluator import ExtendedEvaluator
from ulib.utils.visualize import display_pert

evaluator = ExtendedEvaluator(pert_model, verbose=True)
print(evaluator.evaluate(dl_eval))
display_pert(pert_model)